# Entity Identification Pipeline

Compare the **same model** (`distilbert-base-uncased`) with and without fine-tuning.

**Data Sources:**
- `user_queries.csv`: User questions with entity labels
- `fields_description.csv`: Entity field descriptions (used for both approaches)

**No data leakage**: Test set is never seen during training.

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from typing import List, Dict, Tuple
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss, jaccard_score
)

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "distilbert-base-uncased"
print(f"Device: {DEVICE}, Base model: {BASE_MODEL}")

## 1. Load Data

In [ ]:
# Load both data files
user_queries_df = pd.read_csv('user_queries.csv')
fields_df = pd.read_csv('fields_description.csv')

print(f"User Queries: {len(user_queries_df)} rows")
print(f"Fields Description: {len(fields_df)} rows")
print(f"\nEntities in fields_description: {fields_df['entity_name'].unique().tolist()}")

In [ ]:
# Build entity descriptions from fields_description.csv
def build_entity_descriptions(fields_df: pd.DataFrame) -> Dict[str, str]:
    """Aggregate field descriptions per entity."""
    descriptions = {}
    for entity in fields_df['entity_name'].unique():
        entity_fields = fields_df[fields_df['entity_name'] == entity]
        field_descs = entity_fields['description'].tolist()
        # Create summary: entity name + sample of field descriptions
        descriptions[entity] = f"{entity}. Fields: " + "; ".join(field_descs[:5])
    return descriptions

ENTITY_DESCRIPTIONS = build_entity_descriptions(fields_df)
ALL_ENTITIES = sorted(ENTITY_DESCRIPTIONS.keys())

print(f"Entities ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")
print(f"\nSample description for 'CDR':")
print(ENTITY_DESCRIPTIONS['CDR'][:200] + "...")

In [ ]:
# Extract entities from user queries
def parse_json(json_str: str) -> dict:
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(json_str)
        except:
            return {}

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType."""
    entities = set()
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search(statements):
        if not statements:
            return
        for stmt in statements:
            if isinstance(stmt, dict):
                params = stmt.get('parameters', {})
                if 'relationTargetType' in params:
                    targets = params['relationTargetType']
                    entities.update(targets if isinstance(targets, list) else [targets])
                if 'statements' in stmt:
                    search(stmt['statements'])
    
    search(json_obj.get('statements', []))
    return sorted(list(entities))

user_queries_df['entities'] = user_queries_df['json'].apply(parse_json).apply(extract_entities)

# Distribution
all_flat = [e for ents in user_queries_df['entities'] for e in ents]
print("Entity distribution:")
for entity, count in Counter(all_flat).most_common():
    print(f"  {entity}: {count}")

## 2. Train/Test Split

In [ ]:
# Encode labels and split
mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

X = user_queries_df['question'].tolist()
y = y_encoded

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print("\n*** Test set never used during training ***")

## 3. Shared Components

In [ ]:
class BaseEmbedder:
    """Shared embedding functionality for both approaches."""
    
    def __init__(self, model_name: str = BASE_MODEL):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(DEVICE)
        self.model.eval()
    
    def get_embedding(self, text: str) -> np.ndarray:
        """Get mean-pooled embedding."""
        inputs = self.tokenizer(text, return_tensors='pt', truncation=True, 
                                padding=True, max_length=128)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model(**inputs)
            mask = inputs['attention_mask'].unsqueeze(-1)
            embeddings = outputs.last_hidden_state
            pooled = (embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        
        return pooled.cpu().numpy()[0]
    
    def get_entity_embeddings(self, descriptions: Dict[str, str]) -> Dict[str, np.ndarray]:
        """Get embeddings for all entity descriptions."""
        return {entity: self.get_embedding(desc) for entity, desc in descriptions.items()}


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, 
             labels: List[str], name: str) -> Dict:
    """Evaluate predictions."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
    }
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    for k, v in metrics.items():
        print(f"{k:15}: {v:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=labels, zero_division=0)}")
    
    return metrics

## 4. Approach 1: Pre-trained (No Fine-tuning)

Uses embedding similarity to entity descriptions from `fields_description.csv`.

In [ ]:
class PretrainedClassifier:
    """Pre-trained classifier using embedding similarity. NO fine-tuning."""
    
    def __init__(self, entity_descriptions: Dict[str, str]):
        print(f"Loading pre-trained {BASE_MODEL} (NO fine-tuning)")
        self.embedder = BaseEmbedder(BASE_MODEL)
        self.entities = sorted(entity_descriptions.keys())
        
        # Freeze weights
        for param in self.embedder.model.parameters():
            param.requires_grad = False
        
        # Pre-compute entity embeddings from fields_description
        print("Computing entity embeddings from fields_description.csv...")
        self.entity_embs = self.embedder.get_entity_embeddings(entity_descriptions)
        self.entity_matrix = np.array([self.entity_embs[e] for e in self.entities])
        print(f"Ready. Entity embeddings shape: {self.entity_matrix.shape}")
    
    def predict(self, queries: List[str], threshold: float = 0.5) -> List[List[str]]:
        """Predict via cosine similarity."""
        predictions = []
        for i, query in enumerate(queries):
            if i % 50 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            
            # Get query embedding
            q_emb = self.embedder.get_embedding(query)
            q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-9)
            e_norm = self.entity_matrix / (np.linalg.norm(self.entity_matrix, axis=1, keepdims=True) + 1e-9)
            
            # Cosine similarity
            sims = np.dot(e_norm, q_norm)
            scaled = (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
            
            # Predict above threshold or top-1
            preds = [self.entities[j] for j, s in enumerate(scaled) if s >= threshold]
            if not preds:
                preds = [self.entities[np.argmax(sims)]]
            predictions.append(preds)
        
        return predictions

In [ ]:
# Run Approach 1
print("="*50)
print("APPROACH 1: Pre-trained (No Fine-tuning)")
print("="*50)

pretrained = PretrainedClassifier(ENTITY_DESCRIPTIONS)
preds_pretrained = pretrained.predict(X_test)
y_pred_pretrained = mlb.transform(preds_pretrained)

metrics_pretrained = evaluate(y_test, y_pred_pretrained, ALL_ENTITIES, 
                              "Approach 1: Pre-trained (No Fine-tuning)")

## 5. Approach 2: Fine-tuned

Same model, but trained on our data. Uses entity descriptions as additional context.

In [ ]:
class EntityDataset(Dataset):
    """Dataset for fine-tuning."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }


class FineTunedClassifier:
    """Fine-tuned classifier. Same base model, trained on our data."""
    
    def __init__(self, entity_descriptions: Dict[str, str]):
        self.entities = sorted(entity_descriptions.keys())
        self.descriptions = entity_descriptions
        self.model = None
        self.tokenizer = None
    
    def train(self, X_train: List[str], y_train: np.ndarray, epochs: int = 10):
        """Train on training data only."""
        print(f"\nTraining {BASE_MODEL} on {len(X_train)} samples for {epochs} epochs")
        
        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL, num_labels=len(self.entities),
            problem_type="multi_label_classification"
        )
        
        # Augment training data with entity descriptions as context
        X_aug = []
        for query in X_train:
            # Add condensed entity context
            context = "Entities: " + ", ".join(self.entities)
            X_aug.append(f"{context}. Query: {query}")
        
        # Split for validation (within training data only)
        X_t, X_v, y_t, y_v = train_test_split(X_aug, y_train, test_size=0.15, random_state=42)
        
        train_ds = EntityDataset(X_t, y_t, self.tokenizer)
        val_ds = EntityDataset(X_v, y_v, self.tokenizer)
        
        args = TrainingArguments(
            output_dir='./model_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        trainer = Trainer(
            model=self.model, args=args,
            train_dataset=train_ds, eval_dataset=val_ds
        )
        trainer.train()
        
        self.model.eval()
        self.model.to(DEVICE)
        print("Training complete!")
    
    def predict(self, queries: List[str], threshold: float = 0.5) -> List[List[str]]:
        """Predict with same context format used in training."""
        predictions = []
        context = "Entities: " + ", ".join(self.entities)
        
        for i, query in enumerate(queries):
            if i % 50 == 0:
                print(f"Processing {i+1}/{len(queries)}...")
            
            text = f"{context}. Query: {query}"
            inputs = self.tokenizer(text, return_tensors='pt', truncation=True,
                                    padding=True, max_length=128)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            
            with torch.no_grad():
                logits = self.model(**inputs).logits
                probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            preds = [self.entities[j] for j, p in enumerate(probs) if p > threshold]
            if not preds:
                preds = [self.entities[np.argmax(probs)]]
            predictions.append(preds)
        
        return predictions

In [ ]:
# Run Approach 2
print("="*50)
print("APPROACH 2: Fine-tuned")
print("="*50)

finetuned = FineTunedClassifier(ENTITY_DESCRIPTIONS)
finetuned.train(X_train, y_train, epochs=10)

preds_finetuned = finetuned.predict(X_test)
y_pred_finetuned = mlb.transform(preds_finetuned)

metrics_finetuned = evaluate(y_test, y_pred_finetuned, ALL_ENTITIES,
                             "Approach 2: Fine-tuned")

## 6. Comparison

In [ ]:
print("\n" + "="*60)
print(f"COMPARISON: {BASE_MODEL}")
print("="*60)

comparison = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Hamming Loss'],
    'Pre-trained': [
        metrics_pretrained['exact_match'], metrics_pretrained['f1_micro'],
        metrics_pretrained['f1_macro'], metrics_pretrained['precision'],
        metrics_pretrained['recall'], metrics_pretrained['hamming_loss']
    ],
    'Fine-tuned': [
        metrics_finetuned['exact_match'], metrics_finetuned['f1_micro'],
        metrics_finetuned['f1_macro'], metrics_finetuned['precision'],
        metrics_finetuned['recall'], metrics_finetuned['hamming_loss']
    ]
})

comparison['Δ'] = comparison['Fine-tuned'] - comparison['Pre-trained']
print(comparison.to_string(index=False))

print(f"\nF1 improvement: {metrics_finetuned['f1_micro'] - metrics_pretrained['f1_micro']:+.4f}")
print(f"Exact match improvement: {metrics_finetuned['exact_match'] - metrics_pretrained['exact_match']:+.4f}")

In [ ]:
# Save results
comparison.to_csv('comparison_results.csv', index=False)
print("Results saved to comparison_results.csv")